#**🧠🔍 LlamaIndex RAG Pipeline with OpenAI + Llumo Evaluation**

###**📘 Notebook Overview:**
This notebook implements a Retrieval-Augmented Generation (RAG) workflow using:
- LlamaIndex for loading, indexing, and retrieving from PDF documents
- OpenAI GPT for generating answers
- Llumo SDK for evaluating generated answers using metrics like Context Utilization, Hallucination, etc.


###**📦 1. Install Required Packages**

In [12]:
# 📦 Install LlamaIndex - core framework for building RAG pipelines
!pip install llama-index -q

# 📄 Install LlamaIndex PDF reader - to load and parse PDF files
!pip install llama-index-readers-pdf -q

# This enables LlamaIndex to use OpenAI models like GPT-3.5 and GPT-4 for generating responses
!pip install llama-index-llms-openai -q

# 🤖 Install OpenAI SDK - to interact with GPT models like gpt-3.5-turbo or gpt-4
!pip install openai -q

# 🧪 Install Llumo SDK - to evaluate LLM outputs on metrics like hallucination and context usage
!pip install llumo -q



### **🔐 2. Setup API Keys**

In [3]:
import os

# Set your OpenAI API Key
os.environ["OPENAI_API_KEY"] = "Enter Your Open API Key"

# Set your Llumo API Key
os.environ["LLUMO_API_KEY"] = "Enter Your LLumo Key"

openai_key = os.getenv("OPENAI_API_KEY")
llumo_key = os.getenv("LLUMO_API_KEY")

### **📄 3. Upload and Load PDF Document**

In [5]:
from google.colab import files
from llama_index.readers.file import PDFReader



# ✅ Replace with the actual file name (must match the uploaded file name exactly)
pdf_path = "Howsuccessfulpeoplethink.pdf"  # ⬅️ change this to your actual PDF name

# 📚 Load the PDF using LlamaIndex's PDFReader
reader = PDFReader()
documents = reader.load_data(file=pdf_path)


###**📚 4. Create Index from Documents**


In [6]:
from llama_index.core import VectorStoreIndex
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine(similarity_top_k=3)



###**❓ 5. Ask Questions and Collect Results**

The data used for evaluation will be in the following Example format:
- query: The input question
- context: The contextual data retrieved from the database or other sources to assist in answering the query
- output: The LLM final response as plain text.

```
[  
  {
    "query": "What is the capital of France?",
    "context": "France is a country in Europe. Its capital city is Paris.",
    "output": "The capital of France is Paris."
  },
  {
    "query": "Summarize the plot of 'Romeo and Juliet'.",
    "context": 'Romeo and Juliet' is a tragedy by William Shakespeare.It is about two lovers from feuding families.",
    "output": "Romeo and Juliet is a tragedy by William Shakespeare about two young lovers whose deaths ultimately reconcile their feuding families."
  }
]
```

In [7]:

query_list = [
    "What are the 5 habits of successful people?",
    "How to think big?",
]

results = []
for query in query_list:
    response = query_engine.query(query)
    context_str = "\n".join([node.text for node in response.source_nodes])

    results.append({
        "query": query,
        "context": context_str,
        "output": response.response
    })




In [9]:
results[0]

{'query': 'What are the 5 habits of successful people?',
 'context': 'NOTES\n1.\n James C. Collins and Jerry I. Porras\n, Built to Last: Successful Habits of Visionary\nCompanies\n (New York: Harper Business, 1994), 213.\n2.\n Joshua S. Rubinstein, David E. Meyer, and Jeffrey E. Evans, “Executive Control of Cognitive\nProcesses in Task Switching,” \nJournal of Experimental Psychology\n, quoted in \nLeadership\nStrategies\n, Volume 4, Number 12, December 2001.\n3.\n Annette Moser-Wellman, \nThe Five Faces of Genius: The Skills to Master Ideas at Work\n(New York: Viking, 2001), 6.\n4.\n Annette Moser-Wellman, \nThe Five Faces of Genius: The Skills to Master Ideas at Work\n(New York: Viking, 2001), 9.\n5.\n Ernie J. Zelinski, \nThe Joy of Not Knowing It All: Profiting from Creativity at Work or Play\n(Chicago: VIP Books, 1994), 7.\n6.\n James Allen, \nThe Wisdom of James Allen\n (San Diego: Laurel Creek Press, 1997).\n7.\n Chris Palochko, “Security a Huge Issue at Super Bowl,” sports.yaho

### **🧪 6. Evaluate with Llumo SDK**


In [10]:
# Import the evaluation client from Llumo SDK
from llumo import LlumoClient

# Initialize Llumo client
client = LlumoClient(api_key=llumo_key)

# Evaluate the RAG results
resultDf = client.evaluateMultiple(
    data = results,  # Input Data
    evals = ["Context Utilization", "Redundancy Reduction", "Relevance Retention","Semantic Cohesion","Hallucination"], # List of metrics
    createExperiment = False,   # Set to True to save results as an experiment on the Llumo platform. If False, returns results as a DataFrame or A Python Dict. - Optional
    getDataFrame = True, # Return result as a DataFrame (True) or dictionary (False) - Optional
)




Processing Batches: 100%|██████████| 5/5 [00:16<00:00,  3.30s/batch]


###**📊 7. Evaluate with Llumo SDK**

In [11]:
resultDf

,query,context,output,Context Utilization,Context Utilization Reason,Redundancy Reduction,Redundancy Reduction Reason,Relevance Retention,Relevance Retention Reason,Semantic Cohesion,Semantic Cohesion Reason,Hallucination,Hallucination Reason
0,What are the 5 habits of successful people?,NOTES\n1.\n James C. Collins and Jerry I. Porr...,Successful people have five key habits: \n1. T...,91,The response effectively uses the provided con...,55,The context contains some redundancy through p...,51,The context provides some information related ...,88,"The context mostly maintains a logical flow, b...",35,"The output lists five habits, which are genera..."
1,How to think big?,1\nCultivate Big-Picture Thinking\n“Where succ...,"To think big, one should cultivate big-picture...",89,The response effectively uses the provided con...,57,The text contains some redundancy. For example...,91,The context contains relevant information on h...,93,The context maintains a good logical flow and ...,12,The output is a slight paraphrase of the conte...
